In [ ]:
import os
import sys
import time
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Sto usando il device: {device}")
if device == "cuda":
    print(f"Nome GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Totale: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


repo_name = "BenchmarkingPathologyFoundationModels"
if not os.path.exists(repo_name):
    !git clone https://github.com/colin19950703/BenchmarkingPathologyFoundationModels.git
    print("Repository clonato con successo.")
else:
    print("Repository già presente.")

os.chdir(repo_name)
print(f"Directory di lavoro attuale: {os.getcwd()}")

In [ ]:
# CELLA 2
# Installa le dipendenze del repo
!pip install -r requirements.txt

# Installa librerie extra per il monitoraggio delle risorse (per la tua tesi)
!pip install pynvml matplotlib pandas kaggle

CTransPath

In [ ]:
import os
import requests
import subprocess
import sys

print("🛠️ PASSAGGIO 3: Completamento Installazioni e Download Pesi")


packages = ["loralib", "einops", "transformers", "peft", "tensorboard", "h5py", "openslide-python"]
subprocess.check_call([sys.executable, "-m", "pip", "install"] + packages)


repo_path = os.getcwd() 
pretrained_dir = os.path.join(repo_path, "model_lib", "pretrained")
os.makedirs(pretrained_dir, exist_ok=True)
weights_path = os.path.join(pretrained_dir, "ctranspath.pth")

url = "https://huggingface.co/jamesdolezal/CTransPath/resolve/main/ctranspath.pth"

if not os.path.exists(weights_path):
    try:
        response = requests.get(url, stream=True)
        with open(weights_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        
    except Exception as e:
        print(f"Error downloading weights: {e}")
else:
    print("weights CrtansPath already exist.")

In [ ]:
import os
import cv2
import time
import pandas as pd
import numpy as np
import openslide
from PIL import Image
from tqdm import tqdm


TIFF_DIR = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/PANDA/DATASET/PANDA/images" 

CSV_PATH = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/PANDA/DATASET/PANDA/train.csv"

OUTPUT_DIR = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/PANDA/DATASET/PANDA_TILED_TIFF"


TILE_SIZE = 256

LEVEL = 1         
THRESHOLD_WHITE = 230
MAX_TILES_PER_SLIDE = 100 

# 

def get_tiles(slide_path, level, tile_size, max_tiles):

    slide = openslide.OpenSlide(slide_path)
    

    dims = slide.level_dimensions[level]
    
    tiles = []
    

    downsample_factor = int(slide.level_downsamples[level])
    
    # Coordinate x, y sul livello target
    for y in range(0, dims[1], tile_size):
        for x in range(0, dims[0], tile_size):
            if len(tiles) >= max_tiles:
                break
            
            x0 = x * downsample_factor
            y0 = y * downsample_factor
            
            try:
                tile = slide.read_region((x0, y0), level, (tile_size, tile_size))
                tile = tile.convert("RGB")
                
                gray = tile.convert('L')
                np_gray = np.array(gray)
                white_pixels = np.sum(np_gray > THRESHOLD_WHITE)
                total_pixels = tile_size * tile_size
                
                if (white_pixels / total_pixels) < 0.6:
                    tiles.append((x, y, tile)) 
                    
            except Exception as e:
                continue
                
    slide.close()
    return tiles

# --- MAIN ---
os.makedirs(OUTPUT_DIR, exist_ok=True)


df = pd.read_csv(CSV_PATH)




processed_count = 0
errors = 0

for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing WSI"):
    image_id = row['image_id']
    

    label = str(row['isup_grade']) 
    
    tiff_path = os.path.join(TIFF_DIR, f"{image_id}.tiff")
    

    if not os.path.exists(tiff_path):
        continue
        
    save_dir = os.path.join(OUTPUT_DIR, label)
    os.makedirs(save_dir, exist_ok=True)
    
    try:
        tiles = get_tiles(tiff_path, LEVEL, TILE_SIZE, MAX_TILES_PER_SLIDE)
        
        for (x, y, tile_img) in tiles:
            save_name = f"{image_id}_{x}_{y}.jpg"
            save_path = os.path.join(save_dir, save_name)
            tile_img.save(save_path, "JPEG", quality=90)
            
        processed_count += 1
        
    except Exception as e:
        errors += 1



In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms, models
from tqdm import tqdm
import timm
import gc
from sklearn.model_selection import train_test_split
import types

# 

# 
MODEL_TO_RUN = "ctranspath" 
BATCH_SIZE = 32             
EPOCHS = 10
LR = 1e-4
NUM_CLASSES = 6


TILES_DIR = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/PANDA/DATASET/PANDA_TILED_TIFF"
CSV_PATH = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/PANDA/DATASET/PANDA/train.csv"
CTRANSPATH_WEIGHTS = "./model_lib/pretrained/ctranspath.pth"


Image.MAX_IMAGE_PIXELS = None
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 






df = pd.read_csv(CSV_PATH)



patient_ids = df['image_id'].values
labels_map = df.set_index('image_id')['isup_grade'].to_dict()


train_ids, val_ids = train_test_split(patient_ids, test_size=0.2, random_state=42, stratify=df['isup_grade'])

print(f"Dataset Split:")
print(f"   -> Train: {len(train_ids)}")
print(f"   -> Val:   {len(val_ids)}")


class PandaTileDataset(Dataset):
    def __init__(self, root_dir, patient_ids_list, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        

        valid_ids = set(patient_ids_list)
        

        


        for class_folder in os.listdir(root_dir):
            class_path = os.path.join(root_dir, class_folder)
            if not os.path.isdir(class_path): continue
            

            for img_name in os.listdir(class_path):


                p_id = img_name.split('_')[0]
                

                if p_id in valid_ids:
                    self.image_paths.append(os.path.join(class_path, img_name))
                    self.labels.append(int(class_folder)) 
                    


    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        image = Image.open(path).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
        return image, label


train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), 
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])



train_dataset = PandaTileDataset(TILES_DIR, train_ids, transform=train_transform)

val_dataset = PandaTileDataset(TILES_DIR, val_ids, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)




if MODEL_TO_RUN == "ctranspath":

    def to_2tuple(x): return tuple(x) if isinstance(x, (tuple, list)) else (x, x)

    class ConvStem(nn.Module):
        def __init__(self, img_size=224, patch_size=4, in_chans=3, embed_dim=96, norm_layer=None):
            super().__init__()
            img_size = to_2tuple(img_size)
            patch_size = to_2tuple(patch_size)
            self.stem = nn.Sequential(
                nn.Conv2d(in_chans, embed_dim // 2, kernel_size=3, stride=2, padding=1),
                nn.BatchNorm2d(embed_dim // 2), nn.ReLU(inplace=True),
                nn.Conv2d(embed_dim // 2, embed_dim, kernel_size=3, stride=2, padding=1),
                nn.BatchNorm2d(embed_dim), nn.ReLU(inplace=True),
            )
        def forward(self, x):
            x = self.stem(x)
            x = x.permute(0, 2, 3, 1) 
            return x



    model = timm.create_model(
        "swin_tiny_patch4_window7_224", 
        pretrained=False, 
        embed_dim=128,          
        depths=[2, 2, 18, 2],   
        num_heads=[4, 8, 16, 32]
    )
    

    model.patch_embed = ConvStem(img_size=224, patch_size=4, in_chans=3, embed_dim=128, norm_layer=nn.LayerNorm)
    

    if os.path.exists(CTRANSPATH_WEIGHTS):

        checkpoint = torch.load(CTRANSPATH_WEIGHTS, map_location="cpu")
        if 'model' in checkpoint: checkpoint = checkpoint['model']
        

        model_dict = model.state_dict()
        valid_weights = {k: v for k, v in checkpoint.items() if k in model_dict and model_dict[k].shape == v.shape}
        

        msg = model.load_state_dict(valid_weights, strict=False)

    else:
        print("weights not found!")


    model.head = nn.Linear(model.head.in_features, NUM_CLASSES)
    
    # Fix Pooling
    def pooling_forward(self, x):
        x = self.forward_features(x)
        x = x.mean(dim=[1, 2]) # Global Average Pooling
        x = self.head(x)
        return x
    model.forward = types.MethodType(pooling_forward, model)

elif MODEL_TO_RUN == "phikon":
    try:
        model = timm.create_model("hf_hub:owkin/phikon", pretrained=True, num_classes=NUM_CLASSES)
    except:
        model = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=NUM_CLASSES)

elif MODEL_TO_RUN == "uni":
    try:
        model = timm.create_model("hf-hub:MahmoodLab/UNI2-h", pretrained=True, num_classes=NUM_CLASSES)
    except:

        model = models.resnet50(weights='IMAGENET1K_V1')
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

model = model.to(device)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

print(f"\nSTART TRAINING: {len(train_loader)} batch train, {len(val_loader)} batch val")
history = {'epochs': [], 'loss': [], 'val_acc': []}
best_acc = 0.0

for epoch in range(EPOCHS):
    # --- TRAIN ---
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoca {epoch+1} [Train]"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    # --- val ---
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoca {epoch+1} [Val]"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
    val_acc = 100 * val_correct / val_total
    epoch_loss = running_loss / len(train_loader)
    
    print(f"Epoch {epoch+1}: Train Loss={epoch_loss:.4f} | Val Acc={val_acc:.2f}%")
    
    history['epochs'].append(epoch+1)
    history['loss'].append(epoch_loss)
    history['val_acc'].append(val_acc)
    

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), f"./risultati_finali/{MODEL_TO_RUN}_best.pth")



In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms, models
from tqdm import tqdm
import timm
import gc
from sklearn.model_selection import train_test_split
import types

# 

MODEL_TO_RUN = "ctranspath" 
BATCH_SIZE = 32             
EPOCHS = 10
LR = 1e-4
NUM_CLASSES = 6


TILES_DIR = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/PANDA/DATASET/PANDA_TILED_TIFF"
CSV_PATH = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/PANDA/DATASET/PANDA/train.csv"
CTRANSPATH_WEIGHTS = "./model_lib/pretrained/ctranspath.pth"


Image.MAX_IMAGE_PIXELS = None
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



df = pd.read_csv(CSV_PATH)



patient_ids = df['image_id'].values
labels_map = df.set_index('image_id')['isup_grade'].to_dict()


train_ids, val_ids = train_test_split(patient_ids, test_size=0.2, random_state=42, stratify=df['isup_grade'])

print(f"Dataset Split:")
print(f"   -> Train: {len(train_ids)}")
print(f"   -> Val:   {len(val_ids)}")


class PandaTileDataset(Dataset):
    def __init__(self, root_dir, patient_ids_list, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        

        valid_ids = set(patient_ids_list)
        

        


        for class_folder in os.listdir(root_dir):
            class_path = os.path.join(root_dir, class_folder)
            if not os.path.isdir(class_path): continue
            

            for img_name in os.listdir(class_path):


                p_id = img_name.split('_')[0]
                

                if p_id in valid_ids:
                    self.image_paths.append(os.path.join(class_path, img_name))
                    self.labels.append(int(class_folder)) 
                    


    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        image = Image.open(path).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
        return image, label


train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), 
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])



train_dataset = PandaTileDataset(TILES_DIR, train_ids, transform=train_transform)

val_dataset = PandaTileDataset(TILES_DIR, val_ids, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)






if MODEL_TO_RUN == "ctranspath":

    def to_2tuple(x): return tuple(x) if isinstance(x, (tuple, list)) else (x, x)

    class ConvStem(nn.Module):
        def __init__(self, img_size=224, patch_size=4, in_chans=3, embed_dim=96, norm_layer=None):
            super().__init__()
            img_size = to_2tuple(img_size)
            patch_size = to_2tuple(patch_size)
            self.stem = nn.Sequential(
                nn.Conv2d(in_chans, embed_dim // 2, kernel_size=3, stride=2, padding=1),
                nn.BatchNorm2d(embed_dim // 2), nn.ReLU(inplace=True),
                nn.Conv2d(embed_dim // 2, embed_dim, kernel_size=3, stride=2, padding=1),
                nn.BatchNorm2d(embed_dim), nn.ReLU(inplace=True),
            )
        def forward(self, x):
            x = self.stem(x)
            x = x.permute(0, 2, 3, 1) 
            return x



    model = timm.create_model(
        "swin_tiny_patch4_window7_224", 
        pretrained=False, 
        embed_dim=128,          
        depths=[2, 2, 18, 2],   
        num_heads=[4, 8, 16, 32]
    )
    

    model.patch_embed = ConvStem(img_size=224, patch_size=4, in_chans=3, embed_dim=128, norm_layer=nn.LayerNorm)
    

    if os.path.exists(CTRANSPATH_WEIGHTS):

        checkpoint = torch.load(CTRANSPATH_WEIGHTS, map_location="cpu")
        if 'model' in checkpoint: checkpoint = checkpoint['model']
        

        model_dict = model.state_dict()
        valid_weights = {k: v for k, v in checkpoint.items() if k in model_dict and model_dict[k].shape == v.shape}
        

        msg = model.load_state_dict(valid_weights, strict=False)

    else:
        print("weights not found!")


    model.head = nn.Linear(model.head.in_features, NUM_CLASSES)
    
    # Fix Pooling
    def pooling_forward(self, x):
        x = self.forward_features(x)
        x = x.mean(dim=[1, 2]) # Global Average Pooling
        x = self.head(x)
        return x
    model.forward = types.MethodType(pooling_forward, model)

elif MODEL_TO_RUN == "phikon":
    try:
        model = timm.create_model("hf_hub:owkin/phikon", pretrained=True, num_classes=NUM_CLASSES)
    except:
        model = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=NUM_CLASSES)

elif MODEL_TO_RUN == "uni":
    try:
        model = timm.create_model("hf-hub:MahmoodLab/UNI2-h", pretrained=True, num_classes=NUM_CLASSES)
    except:

        model = models.resnet50(weights='IMAGENET1K_V1')
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

model = model.to(device)


from sklearn.utils.class_weight import compute_class_weight



all_train_labels = [label for _, label in train_dataset]
class_weights = compute_class_weight('balanced', classes=np.unique(all_train_labels), y=all_train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)




criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=LR)

print(f"\nSTART TRAINING: {len(train_loader)} batch train, {len(val_loader)} batch val")
history = {'epochs': [], 'loss': [], 'val_acc': []}
best_acc = 0.0

for epoch in range(EPOCHS):
    # --- TRAIN ---
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoca {epoch+1} [Train]"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    # --- val ---
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoca {epoch+1} [Val]"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
    val_acc = 100 * val_correct / val_total
    epoch_loss = running_loss / len(train_loader)
    
    print(f"Epoch {epoch+1}: Train Loss={epoch_loss:.4f} | Val Acc={val_acc:.2f}%")
    
    history['epochs'].append(epoch+1)
    history['loss'].append(epoch_loss)
    history['val_acc'].append(val_acc)
    

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), f"./risultati_finali/{MODEL_TO_RUN}_best.pth")



In [ ]:
import os
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score, roc_auc_score, f1_score
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import timm
import types
from sklearn.model_selection import train_test_split
from tqdm import tqdm


MODEL_TO_RUN = "ctranspath" 
NUM_CLASSES = 6
BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


TILES_DIR = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/PANDA/DATASET/PANDA_TILED_TIFF"
CSV_PATH = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/PANDA/DATASET/PANDA/train.csv"
CHECKPOINT_PATH = f"./risultati_finali/{MODEL_TO_RUN}_best.pth" # Dove hai salvato il modello


df = pd.read_csv(CSV_PATH)
patient_ids = df['image_id'].values
_, val_ids = train_test_split(patient_ids, test_size=0.2, random_state=42, stratify=df['isup_grade'])

class PandaTileDataset(Dataset):
    def __init__(self, root_dir, patient_ids_list, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        valid_ids = set(patient_ids_list)
        for class_folder in os.listdir(root_dir):
            class_path = os.path.join(root_dir, class_folder)
            if not os.path.isdir(class_path): continue
            for img_name in os.listdir(class_path):
                p_id = img_name.split('_')[0]
                if p_id in valid_ids:
                    self.image_paths.append(os.path.join(class_path, img_name))
                    self.labels.append(int(class_folder)) 

    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        try:
            image = Image.open(self.image_paths[idx]).convert('RGB')
            if self.transform: image = self.transform(image)
            return image, self.labels[idx]
        except: return torch.zeros(3, 224, 224), 0

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_dataset = PandaTileDataset(TILES_DIR, val_ids, transform=val_transform)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)


def to_2tuple(x): return tuple(x) if isinstance(x, (tuple, list)) else (x, x)
class ConvStem(torch.nn.Module):
    def __init__(self, img_size=224, patch_size=4, in_chans=3, embed_dim=96, norm_layer=None):
        super().__init__()
        img_size = to_2tuple(img_size)
        patch_size = to_2tuple(patch_size)
        self.stem = torch.nn.Sequential(
            torch.nn.Conv2d(in_chans, embed_dim // 2, kernel_size=3, stride=2, padding=1),
            torch.nn.BatchNorm2d(embed_dim // 2), torch.nn.ReLU(inplace=True),
            torch.nn.Conv2d(embed_dim // 2, embed_dim, kernel_size=3, stride=2, padding=1),
            torch.nn.BatchNorm2d(embed_dim), torch.nn.ReLU(inplace=True),
        )
    def forward(self, x):
        x = self.stem(x)
        x = x.permute(0, 2, 3, 1) 
        return x

if MODEL_TO_RUN == "ctranspath":

    model = timm.create_model("swin_tiny_patch4_window7_224", pretrained=False, embed_dim=128, depths=[2, 2, 18, 2], num_heads=[4, 8, 16, 32])
    model.patch_embed = ConvStem(img_size=224, patch_size=4, in_chans=3, embed_dim=128, norm_layer=torch.nn.LayerNorm)
    
    model.head = torch.nn.Linear(model.head.in_features, NUM_CLASSES)
    
    def pooling_forward(self, x):
        x = self.forward_features(x)
        x = x.mean(dim=[1, 2])
        x = self.head(x)
        return x
    model.forward = types.MethodType(pooling_forward, model)

model = model.to(DEVICE)

if os.path.exists(CHECKPOINT_PATH):
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
else:
    raise FileNotFoundError(f"weights not found in {CHECKPOINT_PATH}!")


model.eval()
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in tqdm(val_loader):
        images = images.to(DEVICE)
        outputs = model(images)
        probs = torch.nn.functional.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
f1 = f1_score(all_labels, all_preds, average='weighted')
kappa = cohen_kappa_score(all_labels, all_preds, weights='quadratic') # Quadratic Weighted Kappa (Standard per PANDA)

try:
    auc = roc_auc_score(all_labels, all_probs, multi_class='ovr')
except:
    auc = "N/A "

print(f"Accuracy: {accuracy:.4f}")
print(f"F1-Score (Weighted): {f1:.4f}")
print(f"Cohen's Kappa (Quadratic): {kappa:.4f}")
print(f"AUC (One-vs-Rest): {auc}")



report = classification_report(all_labels, all_preds, output_dict=True)
df_report = pd.DataFrame(report).transpose()
df_report.to_csv(f"./risultati_finali/report_completo_{MODEL_TO_RUN}.csv")


In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
import timm
import gc
from sklearn.model_selection import train_test_split
import types


gc.collect()
torch.cuda.empty_cache()


MODEL_TO_RUN = "ctranspath" 
BATCH_SIZE = 32             
EPOCHS = 10
LR = 1e-4
NUM_CLASSES = 6


TILES_DIR = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/PANDA/DATASET/PANDA_TILED_TIFF"
CSV_PATH = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/PANDA/DATASET/PANDA/train.csv"
CTRANSPATH_WEIGHTS = "./model_lib/pretrained/ctranspath.pth"


Image.MAX_IMAGE_PIXELS = None
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



df = pd.read_csv(CSV_PATH)
patient_ids = df['image_id'].values
train_ids, val_ids = train_test_split(patient_ids, test_size=0.2, random_state=42, stratify=df['isup_grade'])

print(f"Dataset Split:")
print(f"   -> Train: {len(train_ids)}")
print(f"   -> Val:   {len(val_ids)}")


class PandaTileDataset(Dataset):
    def __init__(self, root_dir, patient_ids_list, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        valid_ids = set(patient_ids_list)
        

        for class_folder in sorted(os.listdir(root_dir)):
            class_path = os.path.join(root_dir, class_folder)
            if not os.path.isdir(class_path): continue
            for img_name in os.listdir(class_path):
                if '_' in img_name:
                    p_id = img_name.split('_')[0]
                    if p_id in valid_ids:
                        self.image_paths.append(os.path.join(class_path, img_name))
                        self.labels.append(int(class_folder)) 


    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        try:
            image = Image.open(self.image_paths[idx]).convert('RGB')
            if self.transform: image = self.transform(image)
            return image, self.labels[idx]
        except: return torch.zeros(3, 224, 224), 0

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), 
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


train_dataset = PandaTileDataset(TILES_DIR, train_ids, transform=train_transform)

val_dataset = PandaTileDataset(TILES_DIR, val_ids, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)



if MODEL_TO_RUN == "ctranspath":

    def to_2tuple(x): return tuple(x) if isinstance(x, (tuple, list)) else (x, x)

    class ConvStem(nn.Module):
        def __init__(self, img_size=224, patch_size=4, in_chans=3, embed_dim=96, norm_layer=None):
            super().__init__()
            img_size = to_2tuple(img_size)
            patch_size = to_2tuple(patch_size)
            self.proj = nn.Sequential(
                nn.Conv2d(in_chans, embed_dim // 2, kernel_size=3, stride=2, padding=1),
                nn.BatchNorm2d(embed_dim // 2), nn.ReLU(inplace=True),
                nn.Conv2d(embed_dim // 2, embed_dim, kernel_size=3, stride=2, padding=1),
                nn.BatchNorm2d(embed_dim), nn.ReLU(inplace=True),
            )
        def forward(self, x):
            x = self.proj(x)
            x = x.permute(0, 2, 3, 1) 
            return x

    model = timm.create_model(
        "swin_tiny_patch4_window7_224", 
        pretrained=False, 
        embed_dim=128,          
        depths=[2, 2, 18, 2],   
        num_heads=[4, 8, 16, 32]
    )
    model.patch_embed = ConvStem(img_size=224, patch_size=4, in_chans=3, embed_dim=128, norm_layer=nn.LayerNorm)

    if os.path.exists(CTRANSPATH_WEIGHTS):

        checkpoint = torch.load(CTRANSPATH_WEIGHTS, map_location="cpu")
        if 'model' in checkpoint: checkpoint = checkpoint['model']
        if 'state_dict' in checkpoint: checkpoint = checkpoint['state_dict']
        
        model_dict = model.state_dict()
        new_state_dict = {}
        
        for k, v in checkpoint.items():
            new_key = k
            
            new_key = new_key.replace('backbone.', '')
            
            if new_key == 'norm.weight' and 'norm.weight' not in model_dict: new_key = 'head.norm.weight'
            if new_key == 'norm.bias' and 'norm.bias' not in model_dict: new_key = 'head.norm.bias'
            

            if new_key in model_dict and v.shape == model_dict[new_key].shape:
                new_state_dict[new_key] = v
                
        model_dict.update(new_state_dict)
        msg = model.load_state_dict(model_dict, strict=False)
        
        del checkpoint
        del new_state_dict
        gc.collect()
        
    else:
        print("weights not found!")

    model.head = nn.Linear(model.head.in_features, NUM_CLASSES)
    
    def pooling_forward(self, x):
        x = self.forward_features(x)
        x = x.mean(dim=[1, 2]) 
        x = self.head(x)
        return x
    model.forward = types.MethodType(pooling_forward, model)


    for param in model.parameters(): 
        param.requires_grad = False
    for param in model.head.parameters(): 
        param.requires_grad = True

model = model.to(device)

from sklearn.utils.class_weight import compute_class_weight



all_train_labels = train_dataset.labels 

if len(all_train_labels) > 0:
    cw = compute_class_weight('balanced', classes=np.unique(all_train_labels), y=all_train_labels)
    class_weights = torch.tensor(cw, dtype=torch.float).to(device)
    print(f"   -> weights: {class_weights}")
    criterion = nn.CrossEntropyLoss(weight=class_weights)
else:
    criterion = nn.CrossEntropyLoss()

# Optimizer
params_to_update = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.Adam(params_to_update, lr=LR)

print(f"\nSTART TRAINING: {len(train_loader)} batch train, {len(val_loader)} batch val")
history = {'epochs': [], 'loss': [], 'val_acc': []}
best_acc = 0.0

for epoch in range(EPOCHS):
    # TRAIN
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoca {epoch+1} [Train]"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    # VAL
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoca {epoch+1} [Val]"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
    val_acc = 100 * val_correct / val_total
    epoch_loss = running_loss / len(train_loader)
    
    print(f"Epoch {epoch+1}: Train Loss={epoch_loss:.4f} | Val Acc={val_acc:.2f}%")
    
    history['epochs'].append(epoch+1)
    history['loss'].append(epoch_loss)
    history['val_acc'].append(val_acc)
    
    if val_acc > best_acc:
        best_acc = val_acc
        os.makedirs("./risultati_finali", exist_ok=True)
        torch.save(model.state_dict(), f"./risultati_finali/{MODEL_TO_RUN}_best.pth")




PHYKON

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms, models
from tqdm import tqdm
import timm
import gc
from sklearn.model_selection import train_test_split

# 

# 
MODEL_TO_RUN = "phikon"
BATCH_SIZE = 32      #
EPOCHS = 10
LR = 1e-4
NUM_CLASSES = 6


TILES_DIR = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/PANDA/DATASET/PANDA_TILED_TIFF"
CSV_PATH = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/PANDA/DATASET/PANDA/train.csv"
SAVE_DIR = "./risultati_finali"


Image.MAX_IMAGE_PIXELS = None
os.makedirs(SAVE_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 







df = pd.read_csv(CSV_PATH)


patient_ids = df['image_id'].values



train_ids, val_ids = train_test_split(patient_ids, test_size=0.2, random_state=42, stratify=df['isup_grade'])

print(f"Dataset Split:")
print(f"   -> Train: {len(train_ids)}")
print(f"   -> Val:   {len(val_ids)}")


class PandaTileDataset(Dataset):
    def __init__(self, root_dir, patient_ids_list, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        

        valid_ids = set(patient_ids_list)
        

        

        for class_folder in os.listdir(root_dir):
            class_path = os.path.join(root_dir, class_folder)
            if not os.path.isdir(class_path): continue
            

            for img_name in os.listdir(class_path):
                

                p_id = img_name.split('_')[0]
                

                if p_id in valid_ids:
                    self.image_paths.append(os.path.join(class_path, img_name))
                    self.labels.append(int(class_folder)) 
                    


    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        try:
            image = Image.open(path).convert('RGB')
            label = self.labels[idx]
            
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            print(f"Error {path}: {e}")
            return torch.zeros(3, 224, 224), 0


train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), 
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])



train_dataset = PandaTileDataset(TILES_DIR, train_ids, transform=train_transform)

val_dataset = PandaTileDataset(TILES_DIR, val_ids, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)



try:

    model = timm.create_model("hf_hub:owkin/phikon", pretrained=True, num_classes=NUM_CLASSES)
except Exception as e:

    model = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=NUM_CLASSES)

model = model.to(device)


from sklearn.utils.class_weight import compute_class_weight



all_train_labels = [label for _, label in train_dataset]
class_weights = compute_class_weight('balanced', classes=np.unique(all_train_labels), y=all_train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)




criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=LR)

print(f"\nSTART TRAINING: {len(train_loader)} batch train, {len(val_loader)} batch val")
history = {'epochs': [], 'loss': [], 'val_acc': [], 'time': []}
best_acc = 0.0

for epoch in range(EPOCHS):
    start_time = time.time()
    
    # --- TRAIN ---
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoca {epoch+1} [Train]"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    # --- val ---
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoca {epoch+1} [Val]"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
    val_acc = 100 * val_correct / val_total
    epoch_loss = running_loss / len(train_loader)
    duration = time.time() - start_time
    
    print(f"Epoch {epoch+1}: Train Loss={epoch_loss:.4f} | Val Acc={val_acc:.2f}% | Time={duration:.1f}s")
    
    history['epochs'].append(epoch+1)
    history['loss'].append(epoch_loss)
    history['val_acc'].append(val_acc)
    history['time'].append(duration)
    

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, f"{MODEL_TO_RUN}_best.pth"))



UNI

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms, models
from tqdm import tqdm
import timm
import gc
from sklearn.model_selection import train_test_split
from huggingface_hub import login


MODEL_TO_RUN = "uni"
BATCH_SIZE = 8       
EPOCHS = 10
LR = 1e-4
NUM_CLASSES = 6


TILES_DIR = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/PANDA/DATASET/PANDA_TILED_TIFF"
CSV_PATH = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/PANDA/DATASET/PANDA/train.csv"
SAVE_DIR = "./risultati_finali"





Image.MAX_IMAGE_PIXELS = None
os.makedirs(SAVE_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 






df = pd.read_csv(CSV_PATH)
patient_ids = df['image_id'].values


train_ids, val_ids = train_test_split(patient_ids, test_size=0.2, random_state=42, stratify=df['isup_grade'])

print(f"Dataset Split:")
print(f"   -> Train: {len(train_ids)}")
print(f"   -> Val:   {len(val_ids)}")


class PandaTileDataset(Dataset):
    def __init__(self, root_dir, patient_ids_list, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        valid_ids = set(patient_ids_list)
        

        for class_folder in os.listdir(root_dir):
            class_path = os.path.join(root_dir, class_folder)
            if not os.path.isdir(class_path): continue
            
            for img_name in os.listdir(class_path):
                p_id = img_name.split('_')[0]
                if p_id in valid_ids:
                    self.image_paths.append(os.path.join(class_path, img_name))
                    self.labels.append(int(class_folder)) 


    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        try:
            image = Image.open(path).convert('RGB')
            label = self.labels[idx]
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            print(f"Error {path}: {e}")
            return torch.zeros(3, 224, 224), 0

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


train_dataset = PandaTileDataset(TILES_DIR, train_ids, transform=train_transform)

val_dataset = PandaTileDataset(TILES_DIR, val_ids, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)


timm_kwargs = {
    'img_size': 224, 'patch_size': 14, 'depth': 24, 'num_heads': 24, 'init_values': 1e-5, 
    'embed_dim': 1536, 'mlp_ratio': 5.33334, 'num_classes': 0, 'no_embed_class': True,
    'mlp_layer': timm.layers.SwiGLUPacked, 'act_layer': torch.nn.SiLU, 'reg_tokens': 8, 'dynamic_img_size': True
}

class UNIClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = timm.create_model("hf-hub:MahmoodLab/UNI2-h", pretrained=True, **timm_kwargs)

        for param in self.backbone.parameters():
            param.requires_grad = False

        

        self.head = nn.Linear(1536, NUM_CLASSES)
        
    def forward(self, x):
        with torch.no_grad():
            features = self.backbone(x)
        return self.head(features)

try:
    model = UNIClassifier()
    model = model.to(device)

except Exception as e:
    print(f"error UNI: {e}")

    raise e


from sklearn.utils.class_weight import compute_class_weight



all_train_labels = [label for _, label in train_dataset]
class_weights = compute_class_weight('balanced', classes=np.unique(all_train_labels), y=all_train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)




criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.Adam(model.head.parameters(), lr=LR)

print(f"\nSTART TRAINING: {len(train_loader)} batch train, {len(val_loader)} batch val")
history = {'epochs': [], 'loss': [], 'val_acc': [], 'time': []}
best_acc = 0.0

for epoch in range(EPOCHS):
    start_time = time.time()
    
    # --- TRAIN ---
    model.train()
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoca {epoch+1} [Train]"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    # --- VALIDATION ---
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoca {epoch+1} [Val]"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
    val_acc = 100 * val_correct / val_total
    epoch_loss = running_loss / len(train_loader)
    duration = time.time() - start_time
    
    print(f"Epoch {epoch+1}: Train Loss={epoch_loss:.4f} | Val Acc={val_acc:.2f}% | Time={duration:.1f}s")
    
    history['epochs'].append(epoch+1)
    history['loss'].append(epoch_loss)
    history['val_acc'].append(val_acc)
    history['time'].append(duration)
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, f"{MODEL_TO_RUN}_best.pth"))



CONCH

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms, models
from tqdm import tqdm
import timm
import gc
import json
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from huggingface_hub import login



MODEL_TO_RUN = "conch"
BATCH_SIZE = 32      
EPOCHS = 10
LR = 1e-4
NUM_CLASSES = 6


TILES_DIR = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/PANDA/DATASET/PANDA_TILED_TIFF"
CSV_PATH = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/PANDA/DATASET/PANDA/train.csv"
SAVE_DIR = "./risultati_finali"


Image.MAX_IMAGE_PIXELS = None
os.makedirs(SAVE_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



df = pd.read_csv(CSV_PATH)


patient_ids = df['image_id'].values



train_ids, val_ids = train_test_split(patient_ids, test_size=0.2, random_state=42, stratify=df['isup_grade'])

print(f"Dataset Split:")
print(f"   -> Train: {len(train_ids)}")
print(f"   -> Val:   {len(val_ids)}")


class PandaTileDataset(Dataset):
    def __init__(self, root_dir, patient_ids_list, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        

        valid_ids = set(patient_ids_list)
        

        

        for class_folder in os.listdir(root_dir):
            class_path = os.path.join(root_dir, class_folder)
            if not os.path.isdir(class_path): continue
            

            for img_name in os.listdir(class_path):
                

                p_id = img_name.split('_')[0]
                

                if p_id in valid_ids:
                    self.image_paths.append(os.path.join(class_path, img_name))
                    self.labels.append(int(class_folder)) 
                    


    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        try:
            image = Image.open(path).convert('RGB')
            label = self.labels[idx]
            
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            print(f"Error {path}: {e}")
            return torch.zeros(3, 224, 224), 0


train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), 
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])



train_dataset = PandaTileDataset(TILES_DIR, train_ids, transform=train_transform)

val_dataset = PandaTileDataset(TILES_DIR, val_ids, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)



try:
    import open_clip
    from open_clip import factory
    from huggingface_hub import hf_hub_download


    factory._MODEL_CONFIGS['conch_ViT-B-16'] = {
        "embed_dim": 512,
        "vision_cfg": {
            "image_size": 224,
            "layers": 12,
            "width": 768,
            "patch_size": 16,
        },
        "text_cfg": {
            "context_length": 77,
            "vocab_size": 49408,
            "width": 512,
            "heads": 8,
            "layers": 12
        }
    }

    class CONCHClassifier(nn.Module):
        def __init__(self, num_classes):
            super().__init__()
            print("   ⏳ Creazione Architettura CONCH (senza pesi)...")
            
            model, _, _ = open_clip.create_model_and_transforms(
                'conch_ViT-B-16', 
                pretrained=None
            )

            try:

                checkpoint_path = hf_hub_download(
                    repo_id="MahmoodLab/CONCH", 
                    filename="pytorch_model.bin" 
                )
                
                checkpoint = torch.load(checkpoint_path, map_location='cpu')

                if 'state_dict' in checkpoint:
                    checkpoint = checkpoint['state_dict']
                
                model.load_state_dict(checkpoint, strict=False)

                
            except Exception as e:
                print(f"   error downloading weights: {e}")
                print("      Suggerimento: Hai accettato la licenza su https://huggingface.co/MahmoodLab/CONCH ?")
                raise e
            
            self.backbone = model.visual
            

            for param in self.backbone.parameters():
                param.requires_grad = False

            
            self.head = nn.Linear(512, num_classes)
            
        def forward(self, x):
            with torch.no_grad():
                features = self.backbone(x)
            # Gestione output open_clip (a volte è tupla)
            if isinstance(features, tuple):
                features = features[0]
            return self.head(features)

    model = CONCHClassifier(NUM_CLASSES)

except ImportError:
    print("error: missing 'open_clip_torch' or 'huggingface_hub'.")
    raise
except Exception as e:
    print(f"Error CONCH: {e}")
    raise e

model = model.to(device)


all_train_labels = [label for _, label in train_dataset]
class_weights = compute_class_weight('balanced', classes=np.unique(all_train_labels), y=all_train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)




criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.Adam(model.head.parameters(), lr=LR)

print(f"\nSTART TRAINING: {len(train_loader)} batch train, {len(val_loader)} batch val")
history = {'epochs': [], 'loss': [], 'val_acc': [], 'time': []}
best_acc = 0.0

for epoch in range(EPOCHS):
    start_time = time.time()
    
    # --- TRAIN ---
    model.train() 
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoca {epoch+1} [Train]"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    # --- VALIDATION ---
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoca {epoch+1} [Val]"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
    val_acc = 100 * val_correct / val_total
    epoch_loss = running_loss / len(train_loader)
    duration = time.time() - start_time
    
    print(f"Epoch {epoch+1}: Train Loss={epoch_loss:.4f} | Val Acc={val_acc:.2f}% | Time={duration:.1f}s")
    
    history['epochs'].append(epoch+1)
    history['loss'].append(epoch_loss)
    history['val_acc'].append(val_acc)
    history['time'].append(duration)
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, f"{MODEL_TO_RUN}_best.pth"))



VIRCHOW

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
import timm
import gc
import json
import time
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from huggingface_hub import login

# 🧹 PULIZIA MEMORIA
gc.collect()
torch.cuda.empty_cache()


# 
MODEL_TO_RUN = "virchow2"

BATCH_SIZE = 8       
EPOCHS = 10
LR = 1e-4
NUM_CLASSES = 4      


TILES_DIR = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/BACH/TILED_TIFF"
CSV_PATH = "/home/nelloconelli/Escritorio/FOUNDATION MODELS/BACH/ICIAR2018_BACH_Challenge/ICIAR2018_BACH_Challenge/Photos/microscopy_ground_truth.csv"
SAVE_DIR = "./risultati_finali_bach"


Image.MAX_IMAGE_PIXELS = None
os.makedirs(SAVE_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 

print(f"🚀 AVVIO TRAINING SU BACH: {MODEL_TO_RUN.upper()}")




try:
    df = pd.read_csv(CSV_PATH, header=None, names=['filename', 'label_name'])
except:
    print("⚠️ Errore lettura CSV.")


class_map = {'Normal': 0, 'Benign': 1, 'InSitu': 2, 'Invasive': 3,
             'normal': 0, 'benign': 1, 'insitu': 2, 'invasive': 3}
df['label'] = df['label_name'].map(class_map)
df['base_name'] = df['filename'].apply(lambda x: os.path.splitext(x)[0])

patient_ids = df['base_name'].values
labels = df['label'].values

train_ids, val_ids = train_test_split(patient_ids, test_size=0.2, random_state=42, stratify=labels)
print(f"Dataset Split -> Train: {len(train_ids)} | Val: {len(val_ids)}")

class BachTileDataset(Dataset):
    def __init__(self, root_dir, valid_base_names, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        valid_set = set(valid_base_names)
        
        for class_folder in os.listdir(root_dir):
            class_path = os.path.join(root_dir, class_folder)
            if not os.path.isdir(class_path): continue
            for img_name in os.listdir(class_path):
                if not img_name.endswith('.tiff'): continue
                parts = img_name.split('_')
                if len(parts) >= 3:
                    base_name = "_".join(parts[:-2])
                else:
                    base_name = os.path.splitext(img_name)[0]
                
                if base_name in valid_set:
                    self.image_paths.append(os.path.join(class_path, img_name))
                    self.labels.append(int(class_folder))


    def __len__(self): return len(self.image_paths)
    def __getitem__(self, idx):
        try:
            image = Image.open(self.image_paths[idx]).convert('RGB')
            if self.transform: image = self.transform(image)
            return image, self.labels[idx]
        except: return torch.zeros(3, 224, 224), 0

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = BachTileDataset(TILES_DIR, train_ids, transform=train_transform)
val_dataset = BachTileDataset(TILES_DIR, val_ids, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)



class Virchow2Classifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        print("   ⏳ Caricamento Backbone Virchow2 da HuggingFace...")
        
        self.backbone = timm.create_model(
            "hf-hub:paige-ai/Virchow2", 
            pretrained=True, 
            mlp_layer=timm.layers.SwiGLUPacked, 
            act_layer=torch.nn.SiLU,
            num_classes=0,            
            dynamic_img_size=True     
        )
        

        for param in self.backbone.parameters():
            param.requires_grad = False

        self.head = nn.Linear(1280, num_classes)
        
    def forward(self, x):
        with torch.no_grad():
            features = self.backbone(x)
            
        if features.ndim == 3: 
            features = features.mean(dim=1) 
            
        return self.head(features)

try:
    model = Virchow2Classifier(NUM_CLASSES).to(device)
except Exception as e:
    print(f"error loading: {e}")
    raise e


all_train_labels = train_dataset.labels
if len(all_train_labels) > 0:
    cw = compute_class_weight('balanced', classes=np.unique(all_train_labels), y=all_train_labels)
    class_weights = torch.tensor(cw, dtype=torch.float).to(device)
    print(f"   -> Pesi Classi calcolati: {class_weights}")
    criterion = nn.CrossEntropyLoss(weight=class_weights)
else:
    criterion = nn.CrossEntropyLoss()


optimizer = optim.Adam(model.head.parameters(), lr=LR)

print(f"\nSTART TRAINING: {len(train_loader)} batch train, {len(val_loader)} batch val")
history = {'epochs': [], 'loss': [], 'val_acc': [], 'time': []}
best_acc = 0.0

for epoch in range(EPOCHS):
    start_time = time.time()
    
    # --- TRAIN ---
    model.train() 
    running_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoca {epoch+1} [Train]"):
        images = images.to(device)
        labels = labels.to(device).long()
        if labels.dim() > 1: labels = labels.squeeze()
        
        optimizer.zero_grad()
        outputs = model(images) 
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    # --- VALIDATION ---
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoca {epoch+1} [Val]"):
            images = images.to(device)
            labels = labels.to(device).long()
            if labels.dim() > 1: labels = labels.squeeze()
            
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
    val_acc = 100 * val_correct / val_total
    epoch_loss = running_loss / len(train_loader)
    duration = time.time() - start_time
    
    print(f"Epoch {epoch+1}: Train Loss={epoch_loss:.4f} | Val Acc={val_acc:.2f}% | Time={duration:.1f}s")
    
    history['epochs'].append(epoch+1)
    history['loss'].append(epoch_loss)
    history['val_acc'].append(val_acc)
    history['time'].append(duration)
    
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, f"{MODEL_TO_RUN}_best.pth"))
